In [1]:
%pip install syncode transformers torch
%pip install accelerate


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import re
import torch
from syncode import Syncode

extraction_grammar = r"""
start: "Operands: " number " and " number "\nOperator: " operator

number: /[0-9]+(\.[0-9]+)?/
operator: "+" | "-" | "*" | "/"
"""


c:\Users\Soumyadeep Das\Downloads\q2_CS521\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
syn_model = Syncode(model = "microsoft/phi-2", grammar = extraction_grammar, device = "cuda" if torch.cuda.is_available() else "cpu")


[2026-10-27 22:59:53,000-root] - Loading model microsoft/phi-2 with device:cpu, device_map:auto, torch_dtype:torch.bfloat16


Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.90s/it]


In [3]:
FEW_SHORT_PROMPT = """Extract the floating point operands and the binary operator from the query.

Query: What is 327. multiplied by 11.0?
Operands: 327.0 and 11.0
Operator: *

Query: What is 45.1 plus 23.54?
Operands: 45.1 and 23.54
Operator: +

Query: What is 120.4 divided by 4.0?
Operands: 120.4 and 4.0
Operator: /

Query: What is 89.5 minus 12.3?
Operands: 89.5 and 12.3
Operator: -
"""

In [4]:
def run_syncode(textual_input: str):
    """
    Takes a textual math query, runs SynCode for structured extraction, 
    and evaluates the extracted expression using Python.
    
    Returns:
        tuple: (ordered_list_of_operands, operator_str, final_answer_float)
    """
    prompt = f"{FEW_SHORT_PROMPT}\nQuery: {textual_input}\n"
    completions = syn_model.infer(prompt)
    if not completions:
        raise RuntimeError("Syncode returned no completion for the prompt")
    response = completions[0].strip()

    query_operands = re.findall(r"\d+(?:\.\d*)?", textual_input)
    if len(query_operands) != 2:
        raise ValueError(
            f"Expected exactly two numeric operands in the query, found "
            f"{query_operands}: {textual_input!r}"
        )
    op1, op2 = (float(value) for value in query_operands)

    operator_match = re.search(r"Operator:\s*([+\-*/])", response)
    if not operator_match:
        raise ValueError(f"Could not parse an operator from Syncode completion: {response!r}")
    op_str = operator_match.group(1)
        
    operands = [op1, op2]
    
    # Evaluate the binary operation in Python interpreter
    if op_str == "+":
        final_answer = op1 + op2
    elif op_str == "-":
        final_answer = op1 - op2
    elif op_str == "*":
        final_answer = op1 * op2
    elif op_str == "/":
        final_answer = op1 / op2 if op2 != 0 else float("inf")
    else:
        raise ValueError(f"Unsupported operator: {op_str}")
        
    return operands, op_str, float(final_answer)


In [6]:
if __name__ == "__main__":
    test_queries = [
        "What is 200 multiplied by 23.20?",
        "What is 20 plus 10?",
        "What is 4222 divided by 4.0?"
    ]
    
    for query in test_queries:
        operands, operator, answer = run_syncode(query)
        print(f"Query: {query}")
        print(f"  -> Extracted Operands: {operands}")
        print(f"  -> Extracted Operator: {operator}")
        print(f"  -> Final Answer: {answer}\n")

Query: What is 200 multiplied by 23.20?
  -> Extracted Operands: [200.0, 23.2]
  -> Extracted Operator: *
  -> Final Answer: 4640.0

Query: What is 20 plus 10?
  -> Extracted Operands: [20.0, 10.0]
  -> Extracted Operator: +
  -> Final Answer: 30.0

Query: What is 4222 divided by 4.0?
  -> Extracted Operands: [4222.0, 4.0]
  -> Extracted Operator: /
  -> Final Answer: 1055.5

